# Setup

In [1]:
from datasets import load_dataset, Dataset
from transformer_lens import HookedTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import einops
from typing import Literal
from jaxtyping import Float, Int
from torch import Tensor
import functools
import gc
from tqdm import tqdm

import src.data.opi as opi
from src.utils import utils, steering

/Users/paul-philiplouis/work/interpreting-prompt-injection-1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEP_TOKEN = "Data: "

In [13]:
# Loading model
model = HookedTransformer.from_pretrained(MODEL_NAME, device=DEVICE)
tokenizer = model.tokenizer

# Loading dataset
opi_ds = opi.load_opi_dataset()

Loading checkpoint shards: 100%|██████████| 4/4 [00:12<00:00,  3.16s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 224.00 MiB. GPU 0 has a total capacity of 23.69 GiB of which 8.62 MiB is free. Process 3186097 has 23.64 GiB memory in use. Of the allocated memory 23.34 GiB is allocated by PyTorch, and 16.94 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# Useful values
n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads
pattern_layers = [f"blocks.{i}.attn.hook_pattern" for i in range(n_layers)]

In the case of "fixed-form" attacks, the answer, in case of successful injection, is expected under a given form.

For example, if the attack aims at detecting spam (not a very lucrative endeavour for hackers), and the injection explicitely asks to return "spam" or "not spam", injection success can be measured directly in the logits, by looking at the ids of the first token of these answers. In that specific case, "spam" and "not ".

Below, we define a util function to get those ids.

In [ ]:
if model.tokenizer.pad_token is None:
    model.tokenizer.pad_token = model.tokenizer.eos_token
model.tokenizer.padding_side = "left"  # so final position is always index -1

In [ ]:
# ================================================================
# Cross-task train-test layer sweep
# ================================================================
# Logic:
#   For each layer L:
#     train_vector = mean over TRAIN tasks of (mean_resid_combine_L - mean_resid_naive_L)
#     evaluate steering with train_vector on each TEST task's naive prompts
#   Plot mean test-task ASR vs L.
# Expected: plateau in middle layers, drop in late layers as direction
# rotates into task-specific output basis.

CONDITIONS = ["safe", "naive", "escape", "ignore", "combine", "neural_exec", "random"]

task = "sentiment"

prompts = opi.load_opi_per_task(model, task)

N_SUB       = 100
BATCH       = 4
LAYERS      = range(10, n_layers, 4)
COEF        = 1.0                            # × natural mean-shift, per layer

# Checking ASR per condition

In [ ]:
# ================================================================
# Baseline per condition + cache residuals at every layer
# ================================================================
N_SUBSAMPLE = 100   # None = use all. 100 keeps each cell under a few minutes on T4.
BATCH       = 1     # drop to 1 if you hit OOM or attention_mask issues

baseline, residuals = {}, {}
for injection in opi.INJECTIONS:
    baseline[injection] = {}
    residuals[injection] = {}
    pinj = prompts[injection]
    for cond in CONDITIONS:
        plist = pinj['prompts'][cond][:N_SUBSAMPLE] if N_SUBSAMPLE else pinj['prompts'][cond]
        print(f"\n[{injection}][{cond}] : {len(plist)} prompts")
        logits, resids = steering.cache_resid(model, plist, batch_size=BATCH, cache_layers=list(range(0, n_layers, 3)))
        m = steering.compute_metrics(logits, pinj['cor_ids'], pinj['inj_ids'])
        baseline[injection][cond] = m
        residuals[injection][cond] = resids
        print(f"  ASR={m['asr']:.3f}  logit_diff={m['mean_logit_diff']:+.3f}  "
            f"P(inj)={m['mean_p_inj']:.3f}  P(cor)={m['mean_p_cor']:.3f}")

n_inj   = len(opi.INJECTIONS)
n_conds = len(CONDITIONS)
x       = np.arange(n_conds)
width   = 0.8 / n_inj

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for i, injection in enumerate(opi.INJECTIONS):
    offsets = (i - n_inj / 2 + 0.5) * width
    axes[0].bar(x + offsets, [baseline[injection][c]["asr"] for c in CONDITIONS],
                width, label=injection)
    axes[1].bar(x + offsets, [baseline[injection][c]["mean_logit_diff"] for c in CONDITIONS],
                width, label=injection)

for ax in axes:
    ax.set_xticks(x)
    ax.set_xticklabels(CONDITIONS, rotation=30, ha='right')
    ax.legend(title="Injected task")

axes[0].set_title("ASR by condition"); axes[0].set_ylabel("ASR")
axes[1].axhline(0, color="k", lw=0.5)
axes[1].set_title("Mean logit-diff by condition"); axes[1].set_ylabel("logit(inj) − logit(cor)")
plt.tight_layout(); plt.show()

# Per-layer, per-task analysis

In [ ]:
# ================================================================
# Diff-of-means steering vectors
# ================================================================
# Pick the injection to analyze in this section
INJECTION = "spam"   # change to explore other injected tasks

resids = residuals[INJECTION]   # shorthand: {cond: {layer: [N, d_model]}}

steering_combine     = steering.diff_of_means(resids["combine"],     resids["naive"], DEVICE)
steering_neural_exec = steering.diff_of_means(resids["neural_exec"], resids["naive"], DEVICE)

norms_cb = [steering_combine[l].norm().item()     for l in range(n_layers)]
norms_ne = [steering_neural_exec[l].norm().item() for l in range(n_layers)]

plt.figure(figsize=(8, 3))
plt.plot(range(n_layers), norms_cb, marker="o", label="Combine")
plt.plot(range(n_layers), norms_ne, marker="o", label="Neural exec")
plt.xlabel("layer"); plt.ylabel("||v_L||")
plt.title(f"Norm of diff-of-means steering vectors [{INJECTION}]")
plt.tight_layout(); plt.legend(); plt.show()

In [ ]:
cos_sim = [utils.cosine_similarity(steering_combine[l], steering_neural_exec[l]) for l in range(n_layers)]
plt.xlabel("Layer")
plt.ylabel("Cosine similarity")
plt.title("Cosine similarity between combine and neural_exec steering vectors")
plt.plot(cos_sim)

## Steering experiment 

In [ ]:
# ================================================================
# Steering sweep: does adding v_L push 'naive' → 'combine' behavior?
# ================================================================
target_prompts = prompts[INJECTION]["prompts"]["naive"][N_SUBSAMPLE:N_SUBSAMPLE+50]

sweep_layers = list(range(5, n_layers, 4))
sweep_coefs  = [0.5, 1.0, 2.0]

sweep_combine = steering.steering_sweep(
    model, target_prompts, steering_combine,
    sweep_layers, sweep_coefs,
    cor_ids=prompts[INJECTION]["cor_ids"],
    inj_ids=prompts[INJECTION]["inj_ids"],
    batch_size=BATCH,
)
steering.save_results(
    sweep_combine, f"results/{task}_{INJECTION}_combine_sweep.json",
    task=task, injection=INJECTION, sweep_layers=sweep_layers, sweep_coefs=sweep_coefs, rand_baseline=True
)

def _plot_sweep(results, sweep_layers, sweep_coefs, title, axes_row):
    for ax, key, label, cmap in [
        (axes_row[0], "asr", "ASR",        "viridis"),
        (axes_row[1], "ld",  "Logit-diff", "coolwarm"),
    ]:
        im = ax.imshow(results[key], aspect="auto", origin="lower", cmap=cmap)
        ax.set_xticks(range(len(sweep_coefs))); ax.set_xticklabels(sweep_coefs)
        ax.set_yticks(range(len(sweep_layers))); ax.set_yticklabels(sweep_layers)
        ax.set_xlabel("coef"); ax.set_ylabel("layer")
        ax.set_title(f"{title} — {label}"); plt.colorbar(im, ax=ax)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
_plot_sweep({"asr": sweep_combine["asr"],      "ld": sweep_combine["ld"]},
            sweep_layers, sweep_coefs, "Combine vec",    axes[0])
_plot_sweep({"asr": sweep_combine["rand_asr"], "ld": sweep_combine["rand_ld"]},
            sweep_layers, sweep_coefs, "Random control", axes[1])

fig.suptitle(
    f"Steering sweep [{task} / {INJECTION}]  "
    f"naive ASR={baseline[INJECTION]['naive']['asr']:.2f}  "
    f"combine ASR={baseline[INJECTION]['combine']['asr']:.2f}",
    y=1.01,
)
plt.tight_layout(); plt.show()

b = np.unravel_index(sweep_combine["asr"].argmax(), sweep_combine["asr"].shape)
print(f"Best combine: layer={sweep_layers[b[0]]}, coef={sweep_coefs[b[1]]} → ASR={sweep_combine['asr'].max():.3f}")

In [ ]:
# ================================================================
# Refining: does the neural-exec steering vector also steer?
# ================================================================
sweep_layers_fine = list(range(18, n_layers, 2))
sweep_coefs_fine  = [1.0, 2.0]

sweep_neural_exec = steering.steering_sweep(
    model, target_prompts, steering_neural_exec,
    sweep_layers_fine, sweep_coefs_fine,
    cor_ids=prompts[INJECTION]["cor_ids"],
    inj_ids=prompts[INJECTION]["inj_ids"],
    batch_size=BATCH,
)
steering.save_results(
    sweep_neural_exec, f"results/{task}_{INJECTION}_neural_exec_sweep.json",
    task=task, injection=INJECTION, sweep_layers=sweep_layers_fine, sweep_coefs=sweep_coefs_fine, rand_baseline=True
)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
_plot_sweep({"asr": sweep_neural_exec["asr"],      "ld": sweep_neural_exec["ld"]},
            sweep_layers_fine, sweep_coefs_fine, "Neural-exec vec", axes[0])
_plot_sweep({"asr": sweep_neural_exec["rand_asr"], "ld": sweep_neural_exec["rand_ld"]},
            sweep_layers_fine, sweep_coefs_fine, "Random control",  axes[1])

fig.suptitle(
    f"Neural-exec steering sweep [{task} / {INJECTION}]  "
    f"naive ASR={baseline[INJECTION]['naive']['asr']:.2f}  "
    f"neural_exec ASR={baseline[INJECTION]['neural_exec']['asr']:.2f}",
    y=1.01,
)
plt.tight_layout(); plt.show()

b = np.unravel_index(sweep_neural_exec["asr"].argmax(), sweep_neural_exec["asr"].shape)
print(f"Best neural_exec: layer={sweep_layers_fine[b[0]]}, coef={sweep_coefs_fine[b[1]]} → ASR={sweep_neural_exec['asr'].max():.3f}")

# Cross-task tests

In [ ]:
LAYER = 21

hook_name = f"blocks.{LAYER}.hook_resid_post"

def get_mean_resid(prompt_list):
    """Mean residual stream at last token, layer LAYER."""
    resids = []
    for p in prompt_list:
        _, cache = model.run_with_cache(
            p,
            names_filter=lambda n: n == hook_name,
            remove_batch_dim=True
        )
        resids.append(cache[hook_name][-1].detach().cpu().float())
    return torch.stack(resids).mean(dim=0)  # [d_model]


In [ ]:
# ================================================================
# PAIRWISE COSINE MATRIX ACROSS OPI COVER TASKS, GROUPED BY TYPE
# ================================================================

# --- Compute one steering vector per task ---
task_names = ["gigaword", "jfleg", "spam", "hsol", "mrpc", "rte"]
vectors = {}
for task in task_names:
    mean_combine = get_mean_resid(prompts_by_task[task]["combine"])
    mean_naive   = get_mean_resid(prompts_by_task[task]["naive"])
    vectors[task] = mean_combine - mean_naive
    print(f"{task}: vec norm = {vectors[task].norm():.3f}")

# --- Pairwise cosine similarity matrix ---

n = len(task_names)
cos_matrix = np.zeros((n, n))
for i, t1 in enumerate(task_names):
    for j, t2 in enumerate(task_names):
        cos_matrix[i, j] = torch.nn.functional.cosine_similarity(
            vectors[t1].unsqueeze(0), vectors[t2].unsqueeze(0)
        ).item()

# --- Plot ---
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cos_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(task_names, rotation=45, ha="right")
ax.set_yticklabels(task_names)

# Annotate cells
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{cos_matrix[i,j]:.2f}",
                ha="center", va="center", fontsize=9,
                color="white" if abs(cos_matrix[i,j]) > 0.6 else "black")

ax.set_title(f"Pairwise cosine similarity of steering vectors (layer {LAYER})")
plt.colorbar(im, ax=ax, label="Cosine similarity")
plt.tight_layout()
plt.show()


# --- Summary stats ---
off_diag = cos_matrix[np.triu_indices(n, k=1)]
print(f"\nOff-diagonal cosine: mean={off_diag.mean():.3f}, "
      f"std={off_diag.std():.3f}, min={off_diag.min():.3f}, max={off_diag.max():.3f}")

# Steering away